# EDA — sample video (`C3896.MP4`)

Exploratory analysis of the one sample video currently in `samples/` (fixed-camera intersection footage). This is dev/exploratory tooling, not part of the submission interface (`solution.py` doesn't depend on it or on `matplotlib`).

Needs, beyond `requirements.txt`: `pip install matplotlib`.

Two data sources are used:
- **Metadata** (resolution, fps, duration) is probed directly from the real submitted sample, `samples/C3896.MP4` (native 4K).
- **Detections/tracks** for the plots below come from a cached tracker pass over a 120s, 1280x720 resized proxy of the same footage (`dev_data/samples_cache.pkl`, `dev_data/C3896_proxy.mp4`) — reused here to avoid a slow full-resolution re-run just for exploratory plots. The patterns shown (lane geometry, traffic rhythm, pedestrian activity) are representative of the full clip; exact counts would differ slightly on the native 340s video.

In [ ]:
import pickle
import sys
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))  # for unpickling src.detection.Track

NATIVE_VIDEO = REPO_ROOT / "samples" / "C3896.MP4"
PROXY_VIDEO = REPO_ROOT / "dev_data" / "C3896_proxy.mp4"
CACHE = REPO_ROOT / "dev_data" / "samples_cache.pkl"

VEHICLE_CLASSES = {"car", "bus", "truck", "motorcycle", "bicycle"}

## Video metadata

In [ ]:
cap = cv2.VideoCapture(str(NATIVE_VIDEO))
w, h = cap.get(cv2.CAP_PROP_FRAME_WIDTH), cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
fps = cap.get(cv2.CAP_PROP_FPS)
n_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
cap.release()
print(f"{NATIVE_VIDEO.name}: {w:.0f}x{h:.0f} @ {fps:.2f} fps, {n_frames:.0f} frames, {n_frames/fps:.1f}s")

Fixed CCTV-style elevated intersection view, no camera motion — matches the task description exactly ("one angle, no camera motion"). Daytime, clear lighting, no visible weather effects in this clip.

In [ ]:
with open(CACHE, "rb") as fh:
    samples = pickle.load(fh)  # list[(t_sec, list[Track])], src.detection.Track
print(f"cached proxy samples: {len(samples)}, t=[{samples[0][0]:.1f}, {samples[-1][0]:.1f}]s")

cls_counts = Counter()
for _, tracks in samples:
    for tr in tracks:
        cls_counts[tr.cls] += 1
print("raw detection counts by class (not unique tracks):", dict(cls_counts))

## Object counts over time by class

In [ ]:
times = [t for t, _ in samples]
by_class_series = defaultdict(list)
for _, tracks in samples:
    counts = Counter(tr.cls for tr in tracks)
    for c in ("car", "bus", "truck", "person"):
        by_class_series[c].append(counts.get(c, 0))

fig, ax = plt.subplots(figsize=(10, 4))
for c, series in by_class_series.items():
    ax.plot(times, series, label=c, linewidth=1.2)
ax.set_xlabel("time (s)")
ax.set_ylabel("objects in frame")
ax.set_title("Object counts over time by class")
ax.legend()
fig.tight_layout()

Car count oscillates between roughly 15 and 30 in a ~30-40s rhythm — visible surges and troughs consistent with a signal-controlled intersection releasing and re-queuing traffic, not steady free flow. This directly motivated `congestion`'s scene-slow-fraction approach in `src/rules.py` over a naive fixed-count threshold: a fixed threshold would misfire on the natural signal-cycle swing alone.

## Motion heatmap

In [ ]:
heat = np.zeros((72, 128), dtype=np.float64)  # 10px-per-cell grid at 1280x720
for _, tracks in samples:
    for tr in tracks:
        x, y = tr.center
        gx, gy = int(x / 10), int(y / 10)
        if 0 <= gx < 128 and 0 <= gy < 72:
            heat[gy, gx] += 1

fig, ax = plt.subplots(figsize=(8, 4.5))
im = ax.imshow(heat, cmap="inferno", extent=[0, 1280, 720, 0],
                norm=mcolors.LogNorm(vmin=1, vmax=heat.max()))
ax.set_title("Motion heatmap: detection density (log scale)")
fig.colorbar(im, ax=ax, label="detections per 10x10px cell (log)")
fig.tight_layout()

**Pitfall worth recording:** the first version of this plot used a *linear* color scale and came out almost entirely black with just two or three bright pixels. The cause wasn't a bug — it's parked/near-stationary vehicles: a car sitting in the same 10x10px cell for nearly all 899 samples racks up ~840 hits in one cell, which on a linear scale drowns out every moving lane's much smaller per-cell count. Switching to a log color scale (above) reveals the real structure: distinct parallel diagonal streaks matching the road's lanes, exactly matching the trajectory plot below. Lesson for any future occupancy-style heatmap on this footage: always check a linear-scale render for a `parking_zones`-style false read before trusting it.

## Vehicle trajectories

In [ ]:
hist = defaultdict(list)
for t, tracks in samples:
    for tr in tracks:
        if tr.cls in VEHICLE_CLASSES:
            hist[tr.id].append((t, tr.center[0], tr.center[1]))

cap = cv2.VideoCapture(str(PROXY_VIDEO))
cap.set(cv2.CAP_PROP_POS_FRAMES, int(60 * cap.get(cv2.CAP_PROP_FPS)))
ok, frame = cap.read()
cap.release()
bg = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ok else np.zeros((720, 1280, 3), dtype=np.uint8)

long_tracks = [pts for pts in hist.values() if len(pts) >= 15]
rng = np.random.default_rng(0)
sample_idx = rng.choice(len(long_tracks), size=min(40, len(long_tracks)), replace=False)

fig, ax = plt.subplots(figsize=(10, 5.6))
ax.imshow(bg)
for i in sample_idx:
    pts = sorted(long_tracks[i])
    ax.plot([p[1] for p in pts], [p[2] for p in pts], linewidth=1.4, alpha=0.85)
ax.set_title(f"Vehicle trajectories ({len(sample_idx)} of {len(long_tracks)} tracks with >=15 samples)")
ax.set_xlim(0, 1280)
ax.set_ylim(720, 0)
fig.tight_layout()

Two distinct lane bundles are visible: the far carriageway (upper-right, one direction) and the near carriageway (lower, mixing both directions of a two-way boulevard plus a perpendicular side street joining from the left). This mixed-direction geometry is exactly why `wrong_way` was dropped from `CLASSES` (see root `README.md`) — there's no single "correct" heading per zone to compare a track against.

## Traffic density over time

In [ ]:
vehicle_series = [sum(1 for tr in tracks if tr.cls in VEHICLE_CLASSES) for _, tracks in samples]
person_series = [sum(1 for tr in tracks if tr.cls == "person") for _, tracks in samples]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(times, vehicle_series, label="vehicles", color="tab:blue")
ax.plot(times, person_series, label="pedestrians", color="tab:orange")
ax.set_xlabel("time (s)")
ax.set_ylabel("count")
ax.set_title("Traffic density over time")
ax.legend()
fig.tight_layout()

print(f"mean vehicles/frame: {np.mean(vehicle_series):.1f}")
print(f"mean pedestrians/frame: {np.mean(person_series):.1f}")
print(f"unique vehicle tracks: {len(hist)}")
print(f"unique person tracks: {len({tr.id for _, tracks in samples for tr in tracks if tr.cls == 'person'})}")

## Takeaways that shaped the solution

- **Busy, signal-cycled intersection, not free-flowing traffic.** ~23 vehicles/frame on average with a clear periodic surge/trough pattern — `congestion` needed a scene-relative, sustained-fraction test (`src/rules.py`'s `CONGESTION_MIN_FRACTION`/`CONGESTION_MIN_SEC`) rather than a fixed vehicle-count threshold.
- **Mixed-direction carriageway zones** (visible in the trajectory plot) ruled out a simple per-zone "wrong direction" heuristic — directly led to dropping `wrong_way` after measuring per-zone heading concentration (see README).
- **Steady pedestrian presence at crosswalks**, occasionally spiking to 8-10 in frame — motivated `failure_to_yield`'s extra proximity check beyond simple zone co-membership (a busy crossing has *someone* on it most of the time; see `src/rules.py`'s docstring on that rule).
- **Occupancy-style heatmaps need a log scale on this footage** — static/parked vehicles dominate a linear count and hide the actual lane structure (see the Motion heatmap pitfall note above).